In [11]:
"""
COMPLETE SATELLITE LAND COVER ANALYSIS PROJECT
===============================================
MINIMAL DEPENDENCIES VERSION - Works with basic libraries only!
No cv2, no rasterio, no geopandas needed!
"""

import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import re
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Standard image processing - PIL only
from PIL import Image

# ML and stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from scipy import stats, ndimage

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ All imports successful!")


# =============================================================================
# DATA LOADER - Works with any image format
# =============================================================================

class SatelliteDataLoader:
    """
    Load satellite imagery from common formats (PNG, JPG, TIF, etc.)
    Only uses PIL - no specialized libraries!
    """
    
    def __init__(self, base_path: str):
        self.base_path = Path(base_path)
        self.data_inventory = {}
        self.metadata = {}
        
    def scan_directory(self) -> Dict[str, List[Path]]:
        """Scan directory for image files."""
        print("🔍 Scanning directory structure...")
        
        image_extensions = ['.tif', '.tiff', '.png', '.jpg', '.jpeg']
        
        self.data_inventory['images'] = []
        self.data_inventory['json'] = []
        
        for root, dirs, files in os.walk(self.base_path):
            for file in files:
                file_path = Path(root) / file
                ext = file_path.suffix.lower()
                
                if ext in image_extensions:
                    self.data_inventory['images'].append(file_path)
                elif ext in ['.json', '.geojson']:
                    self.data_inventory['json'].append(file_path)
        
        print(f"\n📊 Found {len(self.data_inventory['images'])} images")
        print(f"📊 Found {len(self.data_inventory['json'])} JSON files")
        
        return self.data_inventory
    
    def load_image(self, filepath: Path) -> Tuple[np.ndarray, Dict]:
        """
        Load image - focuses on tifffile for satellite TIFFs
        
        Returns:
            image: numpy array (channels, height, width)
            metadata: dict with image info
        """
        img_array = None
        
        # Method 1: Try tifffile (BEST for satellite multi-band TIFFs)
        try:
            import tifffile
            img_array = tifffile.imread(str(filepath))
            img_array = np.array(img_array, dtype=np.float32)
            
            # Handle different shapes
            if len(img_array.shape) == 2:  # Single band (H, W)
                img_array = img_array[np.newaxis, :, :]
            elif len(img_array.shape) == 3:
                # Check if it's (H, W, C) or (C, H, W)
                if img_array.shape[0] > img_array.shape[2]:  # Likely (H, W, C)
                    img_array = np.transpose(img_array, (2, 0, 1))
                # else already (C, H, W)
            
            # Normalize to 0-1
            if img_array.max() > 1.0:
                img_array = img_array / img_array.max()
            
            metadata = {
                'filename': filepath.name,
                'shape': img_array.shape,
                'dtype': img_array.dtype,
                'path': str(filepath),
                'method': 'tifffile'
            }
            
            return img_array, metadata
            
        except Exception as e:
            print(f"  ✗ tifffile failed for {filepath.name}: {str(e)[:50]}")
        
        # Method 2: Try PIL (fallback)
        try:
            img = Image.open(filepath)
            img_array = np.array(img, dtype=np.float32)
            
            if img_array.max() > 1:
                img_array = img_array / 255.0
            
            if len(img_array.shape) == 2:
                img_array = img_array[np.newaxis, :, :]
            elif len(img_array.shape) == 3 and img_array.shape[2] <= 4:
                img_array = np.transpose(img_array, (2, 0, 1))
            
            metadata = {
                'filename': filepath.name,
                'shape': img_array.shape,
                'dtype': img_array.dtype,
                'path': str(filepath),
                'method': 'PIL'
            }
            
            return img_array, metadata
            
        except Exception as e:
            print(f"  ✗ PIL failed for {filepath.name}: {str(e)[:50]}")
        
        print(f"  ✗ Could not load: {filepath.name}")
        return None, None
    
    def load_all_images(self, max_images: int = 20) -> List[Tuple]:
        """Load multiple images."""
        print(f"\n📷 Loading up to {max_images} images...")
        
        loaded = []
        for i, img_path in enumerate(self.data_inventory['images'][:max_images]):
            img, meta = self.load_image(img_path)
            if img is not None:
                loaded.append((img, meta))
                print(f"  ✓ Loaded {meta['filename']} - shape: {img.shape}")
        
        return loaded


# =============================================================================
# DATA TRANSFORMER
# =============================================================================

class DataTransformer:
    """Clean and transform satellite imagery."""
    
    def __init__(self):
        self.stats = {}
    
    def clean_data(self, data: np.ndarray) -> np.ndarray:
        """Clean data - remove outliers, handle NaN."""
        print("🧹 Cleaning data...")
        cleaned = data.copy().astype(np.float32)
        
        # Remove extreme outliers per band
        for i in range(cleaned.shape[0]):
            band = cleaned[i]
            valid = band[~np.isnan(band)]
            if len(valid) > 0:
                mean, std = valid.mean(), valid.std()
                lower, upper = mean - 3*std, mean + 3*std
                cleaned[i] = np.clip(band, lower, upper)
        
        # Fill NaN
        for i in range(cleaned.shape[0]):
            band = cleaned[i]
            if np.isnan(band).any():
                cleaned[i] = np.nan_to_num(band, nan=np.nanmean(band))
        
        return cleaned
    
    def normalize_bands(self, data: np.ndarray) -> np.ndarray:
        """Normalize to 0-1 range."""
        print("📊 Normalizing bands...")
        normalized = np.zeros_like(data, dtype=np.float32)
        
        for i in range(data.shape[0]):
            band = data[i]
            min_val, max_val = band.min(), band.max()
            if max_val > min_val:
                normalized[i] = (band - min_val) / (max_val - min_val)
            else:
                normalized[i] = band
        
        return normalized
    
    def calculate_ndvi(self, nir_band: np.ndarray, red_band: np.ndarray) -> np.ndarray:
        """Calculate NDVI (vegetation index)."""
        denominator = nir_band + red_band + 1e-8
        ndvi = (nir_band - red_band) / denominator
        return np.clip(ndvi, -1, 1)
    
    def calculate_brightness(self, image: np.ndarray) -> np.ndarray:
        """Calculate brightness from RGB bands."""
        if image.shape[0] >= 3:
            return image[:3].mean(axis=0)
        return image.mean(axis=0)


# =============================================================================
# IMAGE TILER
# =============================================================================

class ImageTiler:
    """Tile large images into smaller patches."""
    
    def __init__(self, tile_size: int = 128, overlap: int = 16):
        self.tile_size = tile_size
        self.overlap = overlap
        self.stride = tile_size - overlap
    
    def create_tiles(self, image: np.ndarray) -> List[Tuple]:
        """
        Create tiles from image.
        
        Returns:
            List of (tile, metadata) tuples
        """
        _, height, width = image.shape
        
        tiles = []
        tile_id = 0
        
        for i in range(0, height - self.tile_size + 1, self.stride):
            for j in range(0, width - self.tile_size + 1, self.stride):
                tile = image[:, i:i+self.tile_size, j:j+self.tile_size]
                
                metadata = {
                    'tile_id': tile_id,
                    'position': (i, j),
                    'shape': tile.shape
                }
                
                tiles.append((tile, metadata))
                tile_id += 1
        
        print(f"✂️  Created {len(tiles)} tiles")
        return tiles
    
    def filter_empty_tiles(self, tiles: List[Tuple], threshold: float = 0.1) -> List[Tuple]:
        """Remove mostly empty/black tiles."""
        filtered = []
        for tile, meta in tiles:
            if np.count_nonzero(tile) / tile.size > threshold:
                filtered.append((tile, meta))
        
        print(f"✓ Kept {len(filtered)}/{len(tiles)} non-empty tiles")
        return filtered


# =============================================================================
# FEATURE EXTRACTOR - No CV2 needed!
# =============================================================================

class FeatureExtractor:
    """Extract features from image tiles using numpy only."""
    
    def __init__(self):
        self.pca = None
        self.feature_names = []
    
    def extract_spectral_features(self, tile: np.ndarray) -> np.ndarray:
        """Extract statistical features from each band."""
        features = []
        
        for band_idx in range(tile.shape[0]):
            band = tile[band_idx].flatten()
            
            features.extend([
                np.mean(band),
                np.std(band),
                np.median(band),
                np.percentile(band, 25),
                np.percentile(band, 75),
                np.min(band),
                np.max(band),
                stats.skew(band),
                stats.kurtosis(band)
            ])
        
        return np.array(features)
    
    def extract_texture_features(self, tile: np.ndarray) -> np.ndarray:
        """Extract simple texture features using numpy only."""
        features = []
        
        # Use first band or average
        if tile.shape[0] >= 3:
            gray = tile[:3].mean(axis=0)
        else:
            gray = tile[0]
        
        # Simple edge detection using Sobel-like filters (no CV2!)
        # Horizontal gradient
        sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
        # Vertical gradient
        sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])
        
        # Apply convolution using scipy
        gx = ndimage.convolve(gray, sobel_x)
        gy = ndimage.convolve(gray, sobel_y)
        
        # Gradient magnitude
        grad_mag = np.sqrt(gx**2 + gy**2)
        
        features.extend([
            np.abs(gx).mean(),
            np.abs(gy).mean(),
            grad_mag.mean(),
            grad_mag.std(),
            grad_mag.max()
        ])
        
        # Laplacian (second derivative)
        laplacian = ndimage.laplace(gray)
        features.extend([
            np.abs(laplacian).mean(),
            laplacian.std()
        ])
        
        # Local variance (texture measure)
        local_var = ndimage.generic_filter(gray, np.var, size=3)
        features.extend([
            local_var.mean(),
            local_var.std()
        ])
        
        return np.array(features)
    
    def extract_all_features(self, tile: np.ndarray) -> np.ndarray:
        """Extract all features from a tile."""
        spectral = self.extract_spectral_features(tile)
        texture = self.extract_texture_features(tile)
        
        return np.concatenate([spectral, texture])
    
    def apply_pca(self, features: np.ndarray, n_components: int = 20) -> np.ndarray:
        """Apply PCA for dimensionality reduction."""
        if self.pca is None:
            n_components = min(n_components, features.shape[1], features.shape[0])
            self.pca = PCA(n_components=n_components)
            reduced = self.pca.fit_transform(features)
            var_explained = self.pca.explained_variance_ratio_.sum()
            print(f"📉 PCA: {features.shape[1]} -> {n_components} features ({var_explained*100:.1f}% variance)")
        else:
            reduced = self.pca.transform(features)
        
        return reduced


# =============================================================================
# CLASSIFIER
# =============================================================================

class DevelopmentClassifier:
    """ML classifier for developed vs undeveloped areas."""
    
    def __init__(self):
        self.model = RandomForestClassifier(
            n_estimators=100,
            max_depth=15,
            random_state=42,
            n_jobs=-1,
            class_weight='balanced'
        )
        self.scaler = StandardScaler()
        self.is_fitted = False
    
    def train(self, X: np.ndarray, y: np.ndarray) -> Dict:
        """Train the model."""
        print("\n🤖 Training Random Forest classifier...")
        
        # Split data
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        # Scale
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_val_scaled = self.scaler.transform(X_val)
        
        # Train
        self.model.fit(X_train_scaled, y_train)
        
        # Evaluate
        train_acc = self.model.score(X_train_scaled, y_train)
        val_acc = self.model.score(X_val_scaled, y_val)
        
        self.is_fitted = True
        
        print(f"✅ Training complete!")
        print(f"  Train accuracy: {train_acc:.4f}")
        print(f"  Val accuracy: {val_acc:.4f}")
        
        return {
            'train_accuracy': train_acc,
            'val_accuracy': val_acc,
            'n_train': len(X_train),
            'n_val': len(X_val)
        }
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Make predictions."""
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Get prediction probabilities."""
        X_scaled = self.scaler.transform(X)
        return self.model.predict_proba(X_scaled)


# =============================================================================
# VISUALIZER
# =============================================================================

class Visualizer:
    """Create visualizations."""
    
    def __init__(self, output_dir: str = "./output/visualizations"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        try:
            plt.style.use('seaborn-v0_8-darkgrid')
        except:
            plt.style.use('default')
    
    def plot_rgb_image(self, image: np.ndarray, title: str = "RGB Image"):
        """Plot RGB composite."""
        fig, ax = plt.subplots(figsize=(10, 8))
        
        if image.shape[0] >= 3:
            rgb = np.transpose(image[:3], (1, 2, 0))
            rgb = np.clip(rgb, 0, 1)
        else:
            rgb = image[0]
        
        ax.imshow(rgb)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')
        
        plt.tight_layout()
        plt.savefig(self.output_dir / 'rgb_composite.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("✓ Saved: rgb_composite.png")
    
    def plot_sample_tiles(self, tiles: List[Tuple], n_samples: int = 8):
        """Plot sample tiles."""
        fig, axes = plt.subplots(2, 4, figsize=(16, 8))
        axes = axes.flatten()
        
        for i, ax in enumerate(axes):
            if i < min(len(tiles), n_samples):
                tile, meta = tiles[i * (len(tiles) // n_samples)]
                
                if tile.shape[0] >= 3:
                    rgb = np.transpose(tile[:3], (1, 2, 0))
                else:
                    rgb = tile[0]
                
                rgb = np.clip(rgb, 0, 1)
                ax.imshow(rgb)
                ax.set_title(f"Tile {meta['tile_id']}", fontsize=10)
                ax.axis('off')
            else:
                ax.axis('off')
        
        plt.suptitle('Sample Image Tiles', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig(self.output_dir / 'sample_tiles.png', dpi=200, bbox_inches='tight')
        plt.close()
        print("✓ Saved: sample_tiles.png")
    
    def plot_confusion_matrix(self, cm: np.ndarray):
        """Plot confusion matrix."""
        fig, ax = plt.subplots(figsize=(8, 6))
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=['Undeveloped', 'Developed'],
                   yticklabels=['Undeveloped', 'Developed'],
                   ax=ax)
        
        ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
        ax.set_ylabel('True', fontsize=12, fontweight='bold')
        ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(self.output_dir / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("✓ Saved: confusion_matrix.png")
    
    def plot_roc_curve(self, fpr, tpr, roc_auc):
        """Plot ROC curve."""
        fig, ax = plt.subplots(figsize=(8, 6))
        
        ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc:.3f})')
        ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random')
        
        ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
        ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
        ax.set_title('ROC Curve', fontsize=14, fontweight='bold')
        ax.legend()
        ax.grid(alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(self.output_dir / 'roc_curve.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("✓ Saved: roc_curve.png")
    
    def plot_feature_importance(self, importances, top_n=20):
        """Plot feature importance."""
        indices = np.argsort(importances)[::-1][:top_n]
        
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.barh(range(top_n), importances[indices], color='steelblue')
        ax.set_yticks(range(top_n))
        ax.set_yticklabels([f'Feature {i}' for i in indices])
        ax.set_xlabel('Importance', fontsize=12, fontweight='bold')
        ax.set_title(f'Top {top_n} Feature Importances', fontsize=14, fontweight='bold')
        ax.invert_yaxis()
        ax.grid(axis='x', alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(self.output_dir / 'feature_importance.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("✓ Saved: feature_importance.png")


# =============================================================================
# MAIN PIPELINE
# =============================================================================

class SatellitePipeline:
    """Complete end-to-end pipeline."""
    
    def __init__(self, data_path: str, output_path: str = "./output"):
        self.data_path = Path(data_path)
        self.output_path = Path(output_path)
        self.output_path.mkdir(parents=True, exist_ok=True)
        
        # Components
        self.loader = SatelliteDataLoader(data_path)
        self.transformer = DataTransformer()
        self.tiler = ImageTiler(tile_size=32, overlap=8)  # SMALLER tiles for small images
        self.feature_extractor = FeatureExtractor()
        self.classifier = DevelopmentClassifier()
        self.visualizer = Visualizer(str(self.output_path / "visualizations"))
        
        self.results = {}
    
    def run(self):
        """Execute full pipeline."""
        print("="*80)
        print("SATELLITE ANALYSIS PIPELINE - STARTING")
        print("="*80)
        
        start_time = datetime.now()
        
        # 1. Load data
        print("\n📥 STAGE 1: Loading Data")
        print("-"*80)
        inventory = self.loader.scan_directory()
        images = self.loader.load_all_images(max_images=20)  # Load MORE images
        
        if not images:
            print("❌ No images found!")
            return
        
        self.results['n_images'] = len(images)
        
        # Combine ALL images into tiles (not just first one)
        all_tiles = []
        
        for img_idx, (image, metadata) in enumerate(images):
            print(f"\n🔄 Processing image {img_idx+1}/{len(images)}: {metadata['filename']}")
            
            # Transform
            cleaned = self.transformer.clean_data(image)
            normalized = self.transformer.normalize_bands(cleaned)
            
            # Create tiles from this image
            tiles = self.tiler.create_tiles(normalized)
            tiles = self.tiler.filter_empty_tiles(tiles, threshold=0.2)
            
            all_tiles.extend(tiles)
        
        print(f"\n✅ Total tiles from all images: {len(all_tiles)}")
        
        if len(all_tiles) < 10:
            print("❌ Not enough tiles! Need at least 10. Try loading more images.")
            return
        
        self.results['n_tiles'] = len(all_tiles)
        
        # Visualize first image
        first_img, first_meta = images[0]
        first_cleaned = self.transformer.clean_data(first_img)
        first_normalized = self.transformer.normalize_bands(first_cleaned)
        self.visualizer.plot_rgb_image(first_normalized, "Original Satellite Image")
        
        # Visualize tiles
        self.visualizer.plot_sample_tiles(all_tiles)
        
        # 4. Extract features
        print("\n🔬 STAGE 4: Feature Extraction")
        print("-"*80)
        features_list = []
        for i, (tile, meta) in enumerate(all_tiles):
            if i % 50 == 0:
                print(f"  Processing tile {i+1}/{len(all_tiles)}")
            features = self.feature_extractor.extract_all_features(tile)
            features_list.append(features)
        
        features = np.array(features_list)
        print(f"✓ Extracted features: {features.shape}")
        
        # PCA
        n_components = min(20, features.shape[1], features.shape[0] // 2)
        features_pca = self.feature_extractor.apply_pca(features, n_components=n_components)
        self.results['n_features'] = features_pca.shape[1]
        
        # 5. Create labels (synthetic for demo)
        print("\n🏷️  STAGE 5: Label Creation")
        print("-"*80)
        labels = self._create_synthetic_labels(all_tiles)
        
        # 6. Train model
        print("\n🎓 STAGE 6: Model Training")
        print("-"*80)
        train_metrics = self.classifier.train(features_pca, labels)
        self.results['train_metrics'] = train_metrics
        
        # 7. Evaluate
        print("\n📊 STAGE 7: Model Evaluation")
        print("-"*80)
        predictions = self.classifier.predict(features_pca)
        proba = self.classifier.predict_proba(features_pca)
        
        # Metrics
        accuracy = accuracy_score(labels, predictions)
        precision = precision_score(labels, predictions)
        recall = recall_score(labels, predictions)
        f1 = f1_score(labels, predictions)
        
        cm = confusion_matrix(labels, predictions)
        fpr, tpr, _ = roc_curve(labels, proba[:, 1])
        roc_auc = auc(fpr, tpr)
        
        print(f"\n{'='*60}")
        print(f"EVALUATION METRICS")
        print(f"{'='*60}")
        print(f"Accuracy:  {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1 Score:  {f1:.4f}")
        print(f"ROC AUC:   {roc_auc:.4f}")
        print(f"{'='*60}\n")
        
        self.results['metrics'] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'roc_auc': roc_auc
        }
        
        # 8. Visualize results
        print("\n📈 STAGE 8: Creating Visualizations")
        print("-"*80)
        self.visualizer.plot_confusion_matrix(cm)
        self.visualizer.plot_roc_curve(fpr, tpr, roc_auc)
        
        if hasattr(self.classifier.model, 'feature_importances_'):
            self.visualizer.plot_feature_importance(self.classifier.model.feature_importances_)
        
        # 9. Generate report
        print("\n📄 STAGE 9: Generating Report")
        print("-"*80)
        self._generate_report()
        
        # Summary
        duration = (datetime.now() - start_time).total_seconds()
        
        print("\n" + "="*80)
        print(f"✅ PIPELINE COMPLETED in {duration:.1f} seconds")
        print("="*80)
        print(f"\n📂 All outputs saved to: {self.output_path}")
        print(f"  • Visualizations: {self.output_path}/visualizations/")
        print(f"  • Report: {self.output_path}/report.txt")
        
        return self.results
    
    def _create_synthetic_labels(self, tiles: List[Tuple]) -> np.ndarray:
        """Create synthetic labels based on tile brightness."""
        labels = []
        
        for tile, meta in tiles:
            brightness = self.transformer.calculate_brightness(tile).mean()
            
            # Simple heuristic: bright = developed
            if brightness > 0.6:
                labels.append(1)  # Developed
            elif brightness < 0.3:
                labels.append(0)  # Undeveloped
            else:
                labels.append(np.random.randint(0, 2))
        
        labels = np.array(labels)
        
        print(f"✓ Created {len(labels)} labels")
        print(f"  Developed: {(labels==1).sum()} ({100*(labels==1).sum()/len(labels):.1f}%)")
        print(f"  Undeveloped: {(labels==0).sum()} ({100*(labels==0).sum()/len(labels):.1f}%)")
        
        return labels
    
    def _generate_report(self):
        """Generate text report."""
        report_path = self.output_path / "report.txt"
        
        report = f"""
{'='*80}
SATELLITE LAND COVER ANALYSIS REPORT
{'='*80}

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Data Source: {self.data_path}

PIPELINE SUMMARY
{'='*80}

1. Data Loading
   - Images loaded: {self.results.get('n_images', 0)}

2. Image Tiling
   - Tiles created: {self.results.get('n_tiles', 0)}
   - Tile size: {self.tiler.tile_size}x{self.tiler.tile_size}

3. Feature Extraction
   - Features per tile: {self.results.get('n_features', 0)}

4. Model Performance
   - Accuracy:  {self.results['metrics']['accuracy']:.4f}
   - Precision: {self.results['metrics']['precision']:.4f}
   - Recall:    {self.results['metrics']['recall']:.4f}
   - F1 Score:  {self.results['metrics']['f1']:.4f}
   - ROC AUC:   {self.results['metrics']['roc_auc']:.4f}

{'='*80}
All visualizations saved to: {self.output_path}/visualizations/
{'='*80}
"""
        
        with open(report_path, 'w') as f:
            f.write(report)
        
        print(f"✓ Report saved to: {report_path}")


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def main():
    """Main function."""
    
    # Configuration - UPDATE THIS PATH!
    DATA_PATH = r"D:\UMBC All Data\DATA 601\SN2_buildings_train_AOI_3_Paris"
    
    # Output will be in the SAME FOLDER as your script
    OUTPUT_PATH = Path.cwd() / "satellite_output"  # Current directory + satellite_output
    
    print(f"""
    ╔══════════════════════════════════════════════════════════════╗
    ║     SATELLITE LAND COVER ANALYSIS - MINIMAL VERSION        ║
    ║          No CV2, Rasterio, or GeoPandas needed!            ║
    ╚══════════════════════════════════════════════════════════════╝
    
    📂 INPUT:  {DATA_PATH}
    📂 OUTPUT: {OUTPUT_PATH}
    
    """)
    
    # Make sure output directory exists and is visible
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
    
    try:
        pipeline = SatellitePipeline(DATA_PATH, str(OUTPUT_PATH))
        results = pipeline.run()
        
        print("\n" + "="*80)
        print("✅ SUCCESS! PIPELINE COMPLETED")
        print("="*80)
        print(f"\n📂 OUTPUT LOCATION:")
        print(f"   {OUTPUT_PATH.absolute()}")
        print(f"\n📁 Files created:")
        print(f"   • {OUTPUT_PATH}/visualizations/rgb_composite.png")
        print(f"   • {OUTPUT_PATH}/visualizations/sample_tiles.png")
        print(f"   • {OUTPUT_PATH}/visualizations/confusion_matrix.png")
        print(f"   • {OUTPUT_PATH}/visualizations/roc_curve.png")
        print(f"   • {OUTPUT_PATH}/visualizations/feature_importance.png")
        print(f"   • {OUTPUT_PATH}/report.txt")
        print(f"\n💡 Open this folder in File Explorer:")
        print(f"   {OUTPUT_PATH.absolute()}")
        print("="*80 + "\n")
        
        # Also print to a simple text file with the location
        location_file = Path.cwd() / "OUTPUT_LOCATION.txt"
        with open(location_file, 'w') as f:
            f.write(f"Your satellite analysis outputs are here:\n\n")
            f.write(f"{OUTPUT_PATH.absolute()}\n\n")
            f.write(f"Files created:\n")
            f.write(f"- visualizations/rgb_composite.png\n")
            f.write(f"- visualizations/sample_tiles.png\n")
            f.write(f"- visualizations/confusion_matrix.png\n")
            f.write(f"- visualizations/roc_curve.png\n")
            f.write(f"- visualizations/feature_importance.png\n")
            f.write(f"- report.txt\n")
        
        print(f"💾 Also saved location to: {location_file.absolute()}\n")
        
        return results
        
    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")
        print(f"\n📂 Check this location for any partial outputs:")
        print(f"   {OUTPUT_PATH.absolute()}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    results = main()

✅ All imports successful!

    ╔══════════════════════════════════════════════════════════════╗
    ║     SATELLITE LAND COVER ANALYSIS - MINIMAL VERSION        ║
    ║          No CV2, Rasterio, or GeoPandas needed!            ║
    ╚══════════════════════════════════════════════════════════════╝
    
    📂 INPUT:  D:\UMBC All Data\DATA 601\SN2_buildings_train_AOI_3_Paris
    📂 OUTPUT: C:\Users\Raunaq\satellite_output
    
    
SATELLITE ANALYSIS PIPELINE - STARTING

📥 STAGE 1: Loading Data
--------------------------------------------------------------------------------
🔍 Scanning directory structure...

📊 Found 4592 images
📊 Found 1148 JSON files

📷 Loading up to 20 images...
  ✓ Loaded MUL_AOI_3_Paris_img10.tif - shape: (8, 162, 162)
  ✓ Loaded MUL_AOI_3_Paris_img100.tif - shape: (8, 162, 162)
  ✓ Loaded MUL_AOI_3_Paris_img1000.tif - shape: (8, 162, 162)
  ✓ Loaded MUL_AOI_3_Paris_img1001.tif - shape: (8, 162, 162)
  ✓ Loaded MUL_AOI_3_Paris_img1003.tif - shape: (8, 162, 162)
  ✓ Lo